In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
import sqlite3
import json

In [ ]:
def init_db():
    conn = sqlite3.connect('restaurant_reviews.db')
    cursor = conn.cursor()
    
    # 리뷰 테이블 생성
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS reviews (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        overall_rating INTEGER,
        review_text TEXT,
        positive_aspects TEXT,
        negative_aspects TEXT
    )
    ''')
    
    # 메뉴 테이블 생성
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS menus (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        review_id INTEGER,
        menu_name TEXT,
        FOREIGN KEY (review_id) REFERENCES reviews (id)
    )
    ''')
    
    # 특징 테이블 생성
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS features (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        review_id INTEGER,
        feature_name TEXT,
        feature_comment TEXT,
        FOREIGN KEY (review_id) REFERENCES reviews (id)
    )
    ''')
    
    conn.commit()
    return conn

In [ ]:
conn = init_db()

In [ ]:
output_parser = JsonOutputParser()

In [ ]:
prompt = ChatPromptTemplate.from_template("""
너는 식당 리뷰를 분석하는 전문가야. 주어진 리뷰에서 다음 정보를 추출해서 JSON 형식으로 제공해줘.

리뷰: {review}

다음 포맷으로 정보를 추출해줘:
```json
{{
    "overall_rating": 1-5 사이의 전체적인 평점(정수),
    "menus": [리뷰에서 언급된 메뉴 목록(배열)],
    "features": [
        {{
            "feature": "특징 이름",
            "comment": "해당 특징에 대한 리뷰 내용, 명사형으로 끝나야 함. 예) ~함. ~라고 생각됨. ~한 부분이 있음"
        }}
    ],
    "positive_aspects": "긍정적인 측면 요약 (없으면 빈 문자열)",
    "negative_aspects": "부정적인 측면 요약 (없으면 빈 문자열)"
}}
```

가능한 메뉴 종류: 파스타, 샐러드, 피자 등
가능한 특징 종류: 맛, 만족도, 서비스, 분위기, 음식량, 메뉴, 목적, 가격, 예약, 대기시간, 주차, 전망, 방역, 위치, 청결도

리뷰에서 명확하게 언급된 정보만 포함시켜주세요.
""")


In [ ]:
llm = ChatOpenAI(model=os.getenv("OPENAI_DEFAULT_MODEL"), temperature=0)
chain = prompt | llm | output_parser

In [ ]:
review = """맛은 있는데 음식이 너무 늦게 나오구 주방에서 일하는 남자 두분이나 마스크 쓰지 않고 서로 웃고 떠들고 하네요
음식이 아무리 맛있어도 청결개념이 부족하면 다시 가고 싶지 않아요
직원교육도 필요할듯하네요"""

result = chain.invoke({"review": review})
print(result)

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute('''
    INSERT INTO reviews (overall_rating, review_text, positive_aspects, negative_aspects)
    VALUES (?, ?, ?, ?)
    ''', (
        result['overall_rating'],
        review,
        result['positive_aspects'],
        result['negative_aspects']
    ))

In [ ]:
review_id = cursor.lastrowid
review_id

In [ ]:
for menu in result['menus']:
    cursor.execute('''
    INSERT INTO menus (review_id, menu_name)
    VALUES (?, ?)
    ''', (review_id, menu))

In [ ]:
for feature in result['features']:
    cursor.execute('''
    INSERT INTO features (review_id, feature_name, feature_comment)
    VALUES (?, ?, ?)
    ''', (review_id, feature['feature'], feature['comment']))

In [ ]:
conn.commit()

In [ ]:
def get_review_info(review_id, conn):
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM reviews WHERE id = ?", (review_id,))
    review_data = cursor.fetchone()
    
    if not review_data:
        return None
    
    cursor.execute("SELECT menu_name FROM menus WHERE review_id = ?", (review_id,))
    menus = [row[0] for row in cursor.fetchall()]
    
    cursor.execute("SELECT feature_name, feature_comment FROM features WHERE review_id = ?", (review_id,))
    features = [{"feature": row[0], "comment": row[1]} for row in cursor.fetchall()]
    
    result = {
        "id": review_data[0],
        "overall_rating": review_data[1],
        "review_text": review_data[2],
        "positive_aspects": review_data[3],
        "negative_aspects": review_data[4],
        "menus": menus,
        "features": features
    }
    
    return result

In [ ]:
result = get_review_info(review_id, conn)
print(result)

In [ ]:
def parse_and_save_review(review_text, conn):
    result = chain.invoke({"review": review_text})
    cursor = conn.cursor()
        
    cursor.execute('''
    INSERT INTO reviews (overall_rating, review_text, positive_aspects, negative_aspects)
    VALUES (?, ?, ?, ?)
    ''', (
        result['overall_rating'],
        review_text,
        result['positive_aspects'],
        result['negative_aspects']
    ))
        
    review_id = cursor.lastrowid
        
    for menu in result['menus']:
        cursor.execute('''
        INSERT INTO menus (review_id, menu_name)
        VALUES (?, ?)
        ''', (review_id, menu))
        
    for feature in result['features']:
        cursor.execute('''
        INSERT INTO features (review_id, feature_name, feature_comment)
        VALUES (?, ?, ?)
        ''', (review_id, feature['feature'], feature['comment']))
        
    conn.commit()
    print(f"리뷰 ID {review_id}가 데이터베이스에 저장되었습니다.")
    return review_id

In [ ]:
review = """평일 런치세트 주문했는데 모든음식이 담백하고 맛있었어요. 
특히 샐러드는 치즈와 야채가 싱싱해서인지 역대급 맛있던 샐러드였고, 이달의 파스타 메뉴였던 감태크림 파스타도 새우와 감태가 듬뿍 들어있는 비싼 고급요리같았어요. 
트러플 풍기 피자도 도우는 바삭하고 치즈는 부드럽고 고소해서 너무 맛있게 먹었어요. 맛없는 음식이 하나도 없었던 뚜에이오 🥹🫰 평일런치세트 뮤조건 드세요!!! 후회없습니다."""

review_id = parse_and_save_review(review, conn)

In [ ]:
result = get_review_info(review_id, conn)
result

In [ ]:
review_text = result['review_text']
del result['review_text']

In [ ]:
result

In [ ]:
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
다음 json data를 읽고 이를 하나의 자연스러운 리뷰로 출력해줘.
                                          
json: {json}

""")

llm = ChatOpenAI(model=os.getenv("OPENAI_DEFAULT_MODEL"), temperature=0)
chain = prompt | llm | StrOutputParser()

In [ ]:
result = chain.invoke({"json": json.dumps(result, ensure_ascii=False)})
print(result)

In [ ]:
print(review_text)